In [1]:
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.ets import AutoETS

from sktime.forecasting.model_selection import ForecastingGridSearchCV
from sktime.split import ExpandingWindowSplitter

warnings.filterwarnings("ignore")

In [2]:
def evaluate_dataset(csv_path: str):
  print("=" * 100)
  print(csv_path)
  print("=" * 100)

  df = pd.read_csv(csv_path)
  df["Date"] = pd.to_datetime(df["Date"])
  df = df.sort_values("Date")
  y = df.set_index("Date")["Revenue"].astype(float)

  y_train = y.loc[:"2010-10-31"]
  y_test = y.loc["2010-11-01":]

  fh = ForecastingHorizon(
    np.arange(1, len(y_test) + 1),
    is_relative=True
  )

  cv = ExpandingWindowSplitter(
    initial_window=int(len(y_train) * 0.60),
    step_length=30,
    fh=np.arange(1, 31)
  )

  print("\nTRAIN SIZE:", len(y_train))
  print("\nTEST SIZE :", len(y_test))

  # =========
  # AUTOARIMA
  # =========
  print("\nSearching AutoARIMA...")

  arima = AutoARIMA(
    seasonal=True,
    suppress_warnings=True,
    error_action="ignore"
  )

  arima_grid = {
    "sp": [7, 30],
    "information_criterion": ["aic", "bic"],
    "max_p": [3, 5],
    "max_q": [3, 5]
  }

  arima_search = ForecastingGridSearchCV(
    forecaster=arima,
    param_grid=arima_grid,
    cv=cv,
    strategy="refit"
  )

  arima_search.fit(y_train)
  best_arima = arima_search.best_forecaster_

  print("\nBEST AUTOARIMA PARAMS")
  print(arima_search.best_params_)

  best_arima.fit(y_train)
  arima_pred = best_arima.predict(fh)

  arima_mae = mean_absolute_error(y_test, arima_pred)
  arima_rmse = np.sqrt(mean_squared_error(y_test, arima_pred))

  # =======
  # AUTOETS
  # =======
  print("Searching AutoETS...")

  ets = AutoETS(auto=True)

  ets_grid = {
    "sp": [7, 30],
    "additive_only": [True, False],
    "allow_multiplicative_trend": [True, False]
  }

  ets_search = ForecastingGridSearchCV(
    forecaster=ets,
    param_grid=ets_grid,
    cv=cv,
    strategy="refit"
  )

  ets_search.fit(y_train)
  best_ets = ets_search.best_forecaster_

  print("\nBEST AUTOETS PARAMS")
  print(ets_search.best_params_)

  best_ets.fit(y_train)
  ets_pred = best_ets.predict(fh)
  ets_mae = mean_absolute_error(y_test, ets_pred)
  ets_rmse = np.sqrt(mean_squared_error(y_test, ets_pred))

  # =======
  # RESULTS
  # =======
  results = pd.DataFrame({
    "Actual": y_test,
    "AutoARIMA": arima_pred,
    "AutoETS": ets_pred
  })

  print("\nFORECASTS")
  print(results)

  print("\nAUTOARIMA")
  print(f"MAE  = {arima_mae:.4f}")
  print(f"RMSE = {arima_rmse:.4f}")

  print("\nAUTOETS")
  print(f"MAE  = {ets_mae:.4f}")
  print(f"RMSE = {ets_rmse:.4f}")


  return {
    "dataset": csv_path,
    "arima_mae": arima_mae,
    "arima_rmse": arima_rmse,
    "ets_mae": ets_mae,
    "ets_rmse": ets_rmse,
    "arima_params": arima_search.best_params_,
    "arima_pred": arima_pred.tolist(),
    "ets_pred": ets_pred.tolist(),
    "n_pred": len(y_test),
    "ets_params": ets_search.best_params_,
  }

In [3]:
import os
os.makedirs('resultados', exist_ok=True)

summary = []
rows = []
for dataset in ["product.csv", "country.csv", "customer.csv"]:
  res = evaluate_dataset("data/" + dataset)
  summary.append(res)
  serie_name = dataset.replace('.csv', '').capitalize()
  if serie_name == 'Country': serie_name = 'País'
  if serie_name == 'Product': serie_name = 'Produto'
  if serie_name == 'Customer': serie_name = 'Cliente'
  
  rows.append({
      'serie': serie_name,
      'periodos_previstos': res['n_pred'],
      'modelo': 'AutoARIMA',
      'valores_previstos': res['arima_pred'],
      'MAE': round(res['arima_mae'], 2),
      'RMSE': round(res['arima_rmse'], 2)
  })
  rows.append({
      'serie': serie_name,
      'periodos_previstos': res['n_pred'],
      'modelo': 'AutoETS',
      'valores_previstos': res['ets_pred'],
      'MAE': round(res['ets_mae'], 2),
      'RMSE': round(res['ets_rmse'], 2)
  })

summary = pd.DataFrame(summary)

df_results = pd.DataFrame(rows)
df_results.to_csv('resultados/resultados_estatisticos.csv', index=False)
print('Resultados salvos em resultados/resultados_estatisticos.csv')

print()
print("=" * 100)
print("FINAL SUMMARY")
print("=" * 100)
print(summary)

data/product.csv

TRAIN SIZE: 165

TEST SIZE : 289

Searching AutoARIMA...


/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x.to_period(freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_


BEST AUTOARIMA PARAMS
{'information_criterion': 'bic', 'max_p': 3, 'max_q': 3, 'sp': 7}


/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x.to_period(freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: 

Searching AutoETS...


...

/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B wi


FORECASTS
              Actual     AutoARIMA       AutoETS
2010-11-01  14076.99  17678.348443  17697.182845
2010-11-02  24263.24  17678.348443  17697.182845
2010-11-03  17411.19  17678.348443  17697.182845
2010-11-04  34311.77  17678.348443  17697.182845
2010-11-05  15652.96  17678.348443  17697.182845
...              ...           ...           ...
2011-12-05  35961.39  17678.348443  17697.182845
2011-12-06  22256.82  17678.348443  17697.182845
2011-12-07  22027.51  17678.348443  17697.182845
2011-12-08  25803.05  17678.348443  17697.182845
2011-12-09   7005.39  17678.348443  17697.182845

[290 rows x 3 columns]

AUTOARIMA
MAE  = 6865.2831
RMSE = 8313.5233

AUTOETS
MAE  = 6872.9468
RMSE = 8319.8215
data/customer.csv

TRAIN SIZE: 335

TEST SIZE : 403

Searching AutoARIMA...

BEST AUTOARIMA PARAMS
{'information_criterion': 'aic', 'max_p': 3, 'max_q': 3, 'sp': 7}
Searching AutoETS...


/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D wi


BEST AUTOETS PARAMS
{'additive_only': True, 'allow_multiplicative_trend': True, 'sp': 7}

FORECASTS
             Actual   AutoARIMA     AutoETS
2010-11-01   716.09  614.725826  408.900913
2010-11-02     0.00  390.916529  409.697067
2010-11-03   944.53  465.068424  410.493221
2010-11-04   942.47  324.777478  411.289375
2010-11-05  1134.10  592.828830  412.085529
...             ...         ...         ...
2011-12-04   713.54  910.243771  725.770273
2011-12-05   359.20  911.250107  726.566427
2011-12-06     0.00  912.253020  727.362581
2011-12-07  3434.99  913.259499  728.158735
2011-12-08   955.71  914.262349  728.954890

[403 rows x 3 columns]

AUTOARIMA
MAE  = 610.6210
RMSE = 672.1948

AUTOETS
MAE  = 522.3947
RMSE = 591.2192
Resultados salvos em resultados/resultados_estatisticos.csv

FINAL SUMMARY
             dataset    arima_mae   arima_rmse      ets_mae     ets_rmse  \
0   data/product.csv    97.824125   120.801155   125.324800   146.548274   
1   data/country.csv  6865.283125  8

/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/pk/studing/projects/machine_learning/revenue_estimate/src/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D wi